<!-- GENERATED from diff-hist/docs/public/competition_reports.md by tools/gen_public_docs.py — edit the source, not here. -->

# Competition Reports

Two dashboards — **Competition Reports for minRTT** and **for Throughput** —
identify other nearby M-Lab servers that outperform your servers to some client
networks. This can be useful when planning peering upgrades.

Each row reflects one triplet ⟨your_site, site2, clientISP⟩ where the path from
M-Lab site2 to the client ISP outperforms the path from your M-Lab site (the
target site) to the same client ISP.

- For **minRTT**, the comparison is the difference between the arithmetic means
  of the minRTTs.
- For **throughput**, the comparison is the ratio of the geometric means of the
  throughputs.

See the **[project overview](https://annealing.mattmathis.net/)** for information about differential histograms and how they expose anomalies in Internet mid-paths.

Full documentation: **[Competition Reports](https://annealing.mattmathis.net/differential-histograms/competition_reports)**.

In [ ]:
# --- Setup ---
import os, sys, json
from datetime import datetime, date, time, timedelta, timezone

# Locate the repo root (directory containing `converter/`) regardless of where
# Voila/Jupyter is launched from.
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "converter")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
from IPython.display import display, HTML, Markdown

from converter import query_builder as qb, runtime as rt
from converter.widget_builder import Controls

client = rt.bq_client()

# Dashboard variable metadata baked in at conversion time.
VARIABLES = json.loads(r"""
[
  {
    "name": "method",
    "type": "custom",
    "label": "Method",
    "description": "Data source backend.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "cached", "value": "cached" },
      { "text": "live", "value": "live" }
    ],
    "current": { "value": "cached" },
    "query_sql": null
  },
  {
    "name": "organization",
    "type": "query",
    "label": "Organization",
    "description": "M-Lab hosting organization.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "All orgs", "value": ".*" }
    ],
    "current": { "value": ".*" },
    "query_sql": "SELECT text, value FROM ( SELECT 'All orgs' AS text, '.*' AS value, 0 AS _sort UNION ALL SELECT DISTINCT  REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') AS text,  REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') AS value,  1 AS _sort FROM `mlab-collaboration.mm_preproduction.cached_metadata` WHERE REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') IS NOT NULL) ORDER BY _sort, text"
  },
  {
    "name": "radius",
    "type": "custom",
    "label": "Radius (kM)",
    "description": "How far should servers be included?",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "1", "value": "1" },
      { "text": "100", "value": "100" },
      { "text": "500", "value": "500" }
    ],
    "current": { "value": "100" },
    "query_sql": "1, 100, 500"
  },
  {
    "name": "ISPcount",
    "type": "custom",
    "label": "Client ISP count",
    "description": "Number of client ISPs to evaluate.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "5", "value": "5" },
      { "text": "10", "value": "10" },
      { "text": "20", "value": "20" },
      { "text": "50", "value": "50" }
    ],
    "current": { "value": "20" },
    "query_sql": "5,10,20, 50"
  }
]
""")


In [ ]:
# --- URL parameter presets (webapp mode) ---
# Voila injects the request query string into os.environ["QUERY_STRING"] before
# executing the notebook.  get_query_string() also handles the preheat-kernel
# case (blocks until the request arrives).  Falls back gracefully in plain
# Jupyter where neither is set.
# For scripted or test overrides, set DASH_PRESETS to a JSON object.
import urllib.parse

url_params = {}
try:
    from voila.utils import get_query_string
    _qs = get_query_string() or ""
    for _k, _vs in urllib.parse.parse_qs(_qs).items():
        url_params[_k] = _vs[0] if len(_vs) == 1 else _vs
except Exception:
    pass

_env = os.environ.get("DASH_PRESETS")
if _env:
    url_params.update(json.loads(_env))

# sites= and ISPs= param handling.
# sites= pre-selects servers by site code (e.g. sites=lga04,lga05).
# ISPs= pre-selects ISPs by AS number (e.g. ISPs=7922,8030).
# If sites= is present but anchor= is not, derive anchor from first site code.
_sites_param = [s.strip() for s in url_params.get('sites', '').split(',') if s.strip()]
_isp_asns    = [s.strip() for s in url_params.get('ISPs',  '').split(',') if s.strip()]
if _sites_param and 'anchor' not in url_params:
    url_params['anchor'] = _sites_param[0][:3]
if _sites_param:
    url_params['region'] = _sites_param


In [ ]:
# --- Dashboard controls (dropdowns; query-backed ones are chained) ---

# End date + duration — ignored when method=cached.
# End defaults to the most recent Sunday (UTC); duration defaults to 7 days.
_today_utc  = datetime.now(timezone.utc).date()
_days_back  = (_today_utc.weekday() + 1) % 7
_end_date   = _today_utc - timedelta(days=_days_back)
w_to       = widgets.DatePicker(value=_end_date, description='End (UTC)',
                                style={"description_width": "90px"})
w_duration = widgets.Dropdown(
    options=[("1 day", 1), ("7 days", 7), ("28 days", 28), ("30 days", 30)],
    value=7, description='Duration',
    style={"description_width": "90px"},
)
w_from = None

# Date row: start/end range (exp) or end + duration (otherwise). Empty when the
# flavor has no date pickers.
if w_from is not None:
    _date_row = widgets.HBox([w_from, w_to], layout=widgets.Layout(margin='2px 0'))
elif w_to is not None and w_duration is not None:
    _date_row = widgets.HBox([w_to, w_duration], layout=widgets.Layout(margin='2px 0'))
else:
    _date_row = widgets.HTML('')

# The method (or methodsrc, in exp) selector drives date-picker visibility; the
# date row is spliced into the controls column right after it.
_method_var = 'methodsrc' if 'methodsrc' in [v['name'] for v in VARIABLES] else 'method'
ctrl = Controls(VARIABLES, client, presets=url_params,
                asn_presets={'ClientISP': _isp_asns} if _isp_asns else None,
                after={_method_var: _date_row})
w_run = widgets.Button(description="Run / Refresh", button_style="primary", icon="play")
_date_label = widgets.HTML('')   # filled from query results after Run

# Hide the date row when the backend token is 'cached'; show it otherwise.
_method_w = ctrl.widgets.get(_method_var)
def _toggle_date_row(*_):
    _is_cached = str(getattr(_method_w, 'value', '')).split('-')[0] == 'cached'
    _date_row.layout.display = 'none' if _is_cached else ''
if _method_w is not None:
    _method_w.observe(_toggle_date_row, names='value')
_toggle_date_row()

# "Extra rows" (if present) is shown only when the selected servers span more
# than one metro (distinct 3-letter IATA prefixes of the site codes).
_extra_w   = ctrl.widgets.get('extra_rows')
_servers_w = ctrl.widgets.get('region')
def _toggle_extra_rows(*_):
    _sel = _servers_w.value if _servers_w is not None else ()
    _metros = {str(s)[:3] for s in _sel}
    _row = getattr(_extra_w, 'widget', _extra_w)
    _row.layout.display = '' if len(_metros) > 1 else 'none'
if _extra_w is not None and _servers_w is not None:
    _servers_w.observe(_toggle_extra_rows, names='value')
    _toggle_extra_rows()


In [ ]:
# --- Competition report panels ---
_DATASET     = "mlab-collaboration.mm_preproduction"
_REPORT_TYPE = "minRTT"   # minRTT or throughput
_X_AXIS      = "none"
_BIN_SIZE    = 50

out = widgets.Output()


def _diagnostics(ctx):
    rows = [(k, ", ".join(v) if isinstance(v, list) else str(v))
            for k, v in ctx.items()]
    return pd.DataFrame(rows, columns=["variable", "value"])


def render(_=None):
    ctx = ctrl.context()
    method  = ctx.get("method", "cached")
    to_dt   = (datetime.combine(w_to.value, time(), tzinfo=timezone.utc)
               if w_to and w_to.value else datetime.now(timezone.utc))
    from_dt = to_dt - timedelta(days=w_duration.value if w_duration else 7)

    out.clear_output(wait=True)
    with out:
        if method == "cached":
            _date_label.value = (
                '<div style="font-size:12px;color:grey;margin:2px 0"><b>Cached data:</b> '
                + rt.get_cached_date_range(client, _DATASET) + '</div>')
        try:
            df = rt.run_competition_report(
                client, _REPORT_TYPE, method,
                ctx.get("organization", ".*"),
                int(ctx.get("radius", 100)),
                int(ctx.get("ISPcount", 5)),
                from_dt, to_dt,
                _DATASET,
            )
        except Exception as exc:
            display(HTML(f"<pre>query failed: {exc}</pre>"))
            df = None

        if df is not None and not df.empty:
            _display_df = df.copy()
            _bc = next((c for c in df.columns if c.lower() == "breadcrumb"), None)
            if _bc:
                _display_df[_bc] = df[_bc].apply(
                    lambda b: f'<a href="{rt.breadcrumb_to_url(str(b))}" target="_blank">{b}</a>'
                    if str(b).strip() else "")
                _display_df = _display_df.rename(columns={_bc: "Breadcrumb"})
            display(widgets.HTML(
                '<div style="height:600px;overflow:auto">'
                + rt.to_html_sticky(_display_df, index=False, na_rep="", escape=False)
                + '</div>'
            ))

        _diag_out = widgets.Output()
        with _diag_out:
            display(_diagnostics(ctx))
        _diag_acc = widgets.Accordion(children=[_diag_out])
        _diag_acc.set_title(0, "Selector Diagnostics")
        _diag_acc.selected_index = None
        display(_diag_acc)


w_run.on_click(render)
if url_params:
    render()


In [ ]:
# --- Display the app ---
# _date_row is inserted inside ctrl.box (right after the method selector) by
# Controls(after=...); only the status label, Run button, and output remain here.
display(widgets.VBox([ctrl.box, _date_label, w_run, out]))
